# 项目 A 实验：跟随一笔趋势目标，走完研究链条

这个实验只追踪一条主线：**2010-03-26 的八个趋势分数，怎样变成第二个交易日使用的多空配置，最后又怎样进入净值和报告？** 我们先逐列复算这一日，再用同一套函数查看完整区间、成本、参数敏感性和失败状态。先把一笔账讲清楚，整条曲线才有可以验证的解释。

默认数据是固定 seed 的合成跨资产价格，不是历史业绩。资产标签只代表不同风险代理，不包含真实股票分红、债券息票或期货到期规则。`src/` 实现是计划权重的最小回放，不包含漂移后实际股数、融资借券、订单簿和真实成交；它用于理解研究方法，不能从一次运行直接推到部署。

运行时选择装有本项目依赖及 `matplotlib` 的 Python 内核，按顺序执行代码。所有数据离线生成，图直接显示；末尾 CLI 对照导出使用系统临时目录，源 Notebook 不保存执行输出。开始前先写一句自己的预期：较短动量窗口会更快反应，但是否也更易被反转和费用伤害？这只是待检验机制，不是承诺。

## 1. 固定问题、对象与实现边界

研究问题是过去价格趋势经过波动缩放、归一化和单名限额后，能否提供相对简单配置的净增量。主配置固定为 60 行动量、20 行调仓、25% 单名上限，先用 10 bps 费用。一个基点是比例数 0.0001，10 bps 意味着每交易 1000 元收 1 元。

时间规则需要读得比箭头更细：第 `t` 行收盘产生目标，回测在 `t` 行记新旧目标变化的费用，第 `t+1` 行才用该目标乘资产从 `t` 收盘到 `t+1` 收盘的收益。这避免让新信号赚到它形成前的同日收益，却仍隐含可以按 `t` 收盘价建立目标的理想成交近似。若实际要等下一开盘，就要重新定义收益区间和成交损失，不能仅凭 `shift(1)` 宣称已经实现真实交易。


In [ ]:
from pathlib import Path
import sys
import tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    candidates = [base, base / "projects/alpha_research", base / "research/projects/alpha_research",
                  base / "run-quant/research/projects/alpha_research"]
    found = [p for p in candidates if p.name == "alpha_research" and (p / "src/backtest.py").exists()]
    if found:
        PROJECT = found[0]
        break
else:
    raise FileNotFoundError("未找到 projects/alpha_research，请从项目或仓库目录运行")
sys.path.insert(0, str(PROJECT))
from src.data import DataConfig, make_prices
from src.features import make_features, signal_to_weights
from src.backtest import run_backtest
from src.metrics import summarize
from src.run import main as run_baseline

config = DataConfig()
prices = make_prices(config)
features = make_features(prices, lookback=60)
weights = signal_to_weights(features["momentum"], features["volatility"], rebalance_every=20, max_abs_weight=0.25)
result = run_backtest(prices, weights, cost_bps=10)
first_target = weights.abs().sum(axis=1).gt(0).idxmax()
first_number = prices.index.get_loc(first_target)
first_return_day = prices.index[first_number + 1]
pd.Series({"first_price_day": prices.index[0], "first_target_day": first_target,
           "first_target_return_day": first_return_day, "assets": len(prices.columns)})


## 2. 从原始价格逐列复算 2010-03-26

合成器先生成三个共同因子，加入资产各自的漂移、噪声及一个很弱的前期收益项，再把对数变化指数累计成正价格。相同 seed 给出相同路径；这个路径拥有研究者已知的生成机制，因此趋势收益最多说明规则怎样响应这些机制，不是发现真实市场规律。

默认第 60 行是 2010-03-26，之前的 60 个日收益已齐备，它也恰好落在每 20 行调仓的格点。**动量（momentum）** 是 `P[t] / P[t-60] - 1`，**滚动年化波动（rolling annualized volatility）** 是最近 60 个日收益的样本标准差乘 `sqrt(252)`。将动量除以波动，得到每个资产的风险缩放分数；由于两者期限单位不同，它只是一个排序与定仓输入，不能叫作预期夏普比率。

以股票代理为例，价格由约 100.078193 变成 95.643664，动量约 -4.431064%；年化波动约 14.200537%；分数约 -0.312035。下面把八个资产都放进一张审计表。`normalized_before_cap` 用分数绝对值和归一化，`target_after_cap` 再限在正负 25%。注意每一行是一个资产，每一列都是从左向右可以复算的一步。


In [ ]:
score = features["momentum"].loc[first_target] / features["volatility"].loc[first_target]
normalizer = score.abs().sum()
audit = pd.DataFrame({
    "price_60_periods_ago": prices.iloc[first_number - 60],
    "price_at_decision": prices.loc[first_target],
    "momentum_60": features["momentum"].loc[first_target],
    "annualized_volatility": features["volatility"].loc[first_target],
    "scaled_score": score,
    "normalized_before_cap": score / normalizer,
    "target_after_cap": weights.loc[first_target],
})
audit.round(6)


股票权重约为 -8.4194%，负号意味着做空方向。这里的权重以自有资本为分母表达名义规模；真实空头还需要可借证券、担保和借券费用，当前实现没有补齐这些机制。

通胀代理在归一化后超过 25%，截断后代码没有重新分配被减掉的余量，所以总绝对权重约 0.954179，而不是始终等于 1。净权重约 -0.153115，说明多空金额合计偏空，但“净空 15.3%”不代表风险只有 15.3%：不同资产的波动和共同因子不同，仍有约 95.4% 的总名义规模。

### 手工停顿：在运行下一格前推一次收益

第二个交易日股票价格约为 95.922607，相对 95.643664 上涨约 0.29165%。负的股票目标乘正的股票收益，应给组合负贡献，大约 -0.024555%。其余七项合计是否足够抵消它，只有逐项相加后才知道。

<details><summary>参考答案</summary>

股票贡献为 `-0.08419405 × (95.922607 / 95.643664 - 1)`，约 -0.00024555，即 -2.4555 个基点。单一空头亏损不意味着组合必亏；八项贡献合计毛收益约 0.001672413，即 +16.7241 个基点。下面表中的数值使用未四舍五入价格，因此更精确。

</details>


In [ ]:
next_day_audit = pd.DataFrame({
    "previous_target": weights.loc[first_target],
    "asset_return_next_day": features["returns"].loc[first_return_day],
})
next_day_audit["contribution_to_portfolio"] = next_day_audit.previous_target * next_day_audit.asset_return_next_day
print(next_day_audit.round(7))
print("贡献之和：", next_day_audit.contribution_to_portfolio.sum())
print("回测毛收益：", result.loc[first_return_day, "gross_return"])
result.loc[first_target:first_return_day, ["gross_return", "turnover", "cost", "net_return", "equity"]]


3 月 26 日之前目标为零，所以该日毛收益为零；但新目标总规模约 0.954179，以 10 bps 收费，费用为 `0.954179 × 10 / 10000 ≈ 0.000954179`，净值先降到约 0.999045821。3 月 29 日目标未变，计划差换手为零，净收益等于刚算的毛收益；它是在已经扣费的净值上复利，所以新净值约 1.000716639。

这是一条可以观察的时间链：**先得到信息和目标、扣调仓费，再让这个目标解释下一行价格变化**。如果把权重再移一次，就推迟了策略；如果不移，就让目标利用同日已经发生的收益。审阅时应先对齐每一列定义，不能只凭代码里出现 `shift` 就判断正确。

### 固定目标不是固定份数

在两个调仓日之间，代码向前延用目标比例，每天用相同比例乘当天收益，却不模拟份数和市值漂移，也不计恢复固定比例的费用。例如股票上涨后，现实恒份额组合里的股票比例会变大；这里下一日仍使用原目标比例。`weights.diff()` 为零只代表计划没变，不代表真实账户不需要交易。

因此本项目是固定目标比例的收益与计划变更成本近似，不能称为完整自融资回测。现金利息、融资、借券、保证金、实际成交延迟和漂移交易均未建模。第二章金额账本可以作为未来扩展方向；当前先保持最小实现，让每个近似都能被指出。

## 3. 看整条路径：先解释差异，再评价好坏

我们现在只放大观察范围，不改变刚才的规则。四条趋势净值来自同一份目标，唯一不同是单位交易额费率；第五条是八资产每期 1/8 的等权固定比例基准。基准首行从零建立目标，所以付一次总规模为 1 的建仓费，之后没有目标变化费用；它也未计算漂移再平衡，并非恒份额买入持有。

预期应该是什么？在同一目标下，提高非负费用会逐日降低净收益、压低净值；它不会改变目标和换手。趋势策略不保证胜过等权，它的多空、净暴露和波动也不同。若趋势落后，先检查机制和风险差异，不能直接把窗口换成图上更漂亮的一组。


In [ ]:
benchmark_weights = pd.DataFrame(1 / len(prices.columns), index=prices.index, columns=prices.columns)
benchmark = run_backtest(prices, benchmark_weights, cost_bps=10)
cost_results = {c: run_backtest(prices, weights, cost_bps=c) for c in [0, 10, 25, 50]}
curves = {f"trend_{c}bps": r.equity for c, r in cost_results.items()}
curves["equal_weight_10bps"] = benchmark.equity
ax = pd.DataFrame(curves).plot(figsize=(11, 5), logy=True, title="Synthetic fixed-target replay: net wealth")
ax.set_ylabel("wealth, log scale")
ax.grid(alpha=0.25)
plt.show()

baseline_report = pd.DataFrame({f"trend_{c}bps": summarize(r.net_return) for c, r in cost_results.items()}).T
baseline_report.loc["equal_weight_10bps"] = pd.Series(summarize(benchmark.net_return))
baseline_report.round(5)


报告中的 `annual_return` 是从整条净值连乘得到的复合年化增长；`sharpe` 是日均净收益除日标准差再乘 `sqrt(252)`，现金基准在教学里设为零；`max_drawdown` 为负数，包含初始净值 1。它们分别描述增长、单位一般波动回报和历史峰值缺口，不能互相替代。

全区间报告还包括热身期：趋势等到 3 月才建仓，基准从 1 月就配置，二者并非从第一天起就同等暴露。若研究想比较从首次有效信号之后的表现，可以在共同起点重新建两份计划，明确都支付当日建仓费。下面这种比较仍有不同 beta 和波动，不会自动变成 alpha 识别，但至少把评估期间对齐了。


In [ ]:
common_prices = prices.loc[first_target:]
common_trend = run_backtest(common_prices, weights.loc[first_target:], cost_bps=10)
common_benchmark = run_backtest(common_prices, benchmark_weights.loc[first_target:], cost_bps=10)
pd.DataFrame({"trend_common_start": summarize(common_trend.net_return),
              "equal_weight_common_start": summarize(common_benchmark.net_return)}).T.round(5)


## 4. 暴露审计：为什么不能只看收益表

**总暴露（gross exposure）** 是绝对权重之和，**净暴露（net exposure）** 是带符号权重之和；最大单名权重检查限额，非零资产数只统计参与数量，不能当作独立机会数。八个合成标签共享三个因子，因此可能同时亏损。

原始分数先归一化到总绝对权重 1，再截断单名，上限会让总暴露低于 1。它没有让“每个资产贡献相同风险”，更没有实现市场中性。预期图中最大单名不超过 0.25，热身期全零，之后只有计划调仓格点出现阶梯变化。若日内突然出现很高峰值，应先检查缺失、索引和截断，而不是用“市场变化”解释代码约束失效。


In [ ]:
exposure = pd.DataFrame({
    "gross": weights.abs().sum(axis=1),
    "net": weights.sum(axis=1),
    "largest_abs_name": weights.abs().max(axis=1),
    "nonzero_assets": weights.ne(0).sum(axis=1),
})
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
exposure[["gross", "net"]].plot(ax=axes[0], title="Planned exposures")
exposure.largest_abs_name.plot(ax=axes[1], color="#d97706", title="Largest absolute target weight")
axes[1].axhline(.25, color="black", linestyle="--")
fig.tight_layout()
plt.show()
print(exposure.loc[first_target].round(6))


## 5. 参数与费用：一次改变一个解释

这里比较 20、60、120 行动量窗口，调仓间隔、上限、数据和费用集合固定。较短窗口可能反应更快，也可能在震荡中频繁翻向；较长窗口可能稳定，也可能来不及退出反转。**没有任何一个方向保证净收益更高。** 先比较这些预期是否在分数、换手和净结果中同时出现。

窗口还会改变热身长度。为了不让“20 行更早建仓”混入窗口比较，下面先在完整历史上构建计划和回放，再只评价 2012 年之后的共同区间。评估起点继承此前已建好的计划，不凭空重新建仓，所以这里测的是持续运行策略在该段的表现，不是 2012 年新账户启动。

图上的 12 个组合都是事后展示，不构成盲测。如果看过它们再挑最好窗口，同一份区间就不能再用来声称未经选择的样本外成功。真实研究应提前分配训练、验证和最后测试期，只在允许的区间作选择。


In [ ]:
rows = []
for lookback in [20, 60, 120]:
    local_features = make_features(prices, lookback=lookback)
    local_weights = signal_to_weights(local_features["momentum"], local_features["volatility"], rebalance_every=20)
    for cost_bps in [0, 10, 25, 50]:
        local_result = run_backtest(prices, local_weights, cost_bps=cost_bps).loc["2012":]
        row = summarize(local_result.net_return)
        row.update(lookback=lookback, cost_bps=cost_bps, average_turnover=local_result.turnover.mean())
        rows.append(row)
sensitivity = pd.DataFrame(rows)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lookback, group in sensitivity.groupby("lookback"):
    axes[0].plot(group.cost_bps, group.annual_return, marker="o", label=f"lookback={lookback}")
    axes[1].plot(group.cost_bps, group.average_turnover, marker="o", label=f"lookback={lookback}")
axes[0].set(title="Net annual return, common evaluation", xlabel="cost bps", ylabel="annual return")
axes[1].set(title="Target-change turnover", xlabel="cost bps", ylabel="daily mean turnover")
for ax in axes:
    ax.legend()
    ax.grid(alpha=.25)
fig.tight_layout()
plt.show()
sensitivity.pivot(index="lookback", columns="cost_bps", values="sharpe").round(4)


结果判断分三层。第一，固定窗口的费用线必须体现单期收益随费率下降，换手线应水平，因为此模型没有费用反馈到决策。第二，看短窗口是否真的比长窗口更常改变目标；如果没有，调仓格点和单名上限可能抑制了差异。第三，最后才看净收益是否有足够余量，而不是挑一列最大的比率。

夏普并不要求随费率严格单调：费率既影响均值，也改变某些日子的波动，分母会动。能严格推断的是同一目标、非负换手下，单期净收益随成本上升不增加。不能把对净收益的不变量随手扩大成对所有摘要统计的断言。

## 6. 归因与失败：把净值变化拆回每一项

同一回放可精确拆成资产权重乘收益的贡献，减去成本。为了跨期相加，我们将每日收益贡献乘该日之前的净值，得到以初始资本为单位的金额贡献；这些金额相加才应等于最终净值减 1。直接把日收益率贡献累加，不能精确解释复合终值。

这仍然是当前模型内的损益归因，不是真实成交归因。若 `CRYPTO` 一项主导所有收益，结论可能只适用于合成高波动代理；若多空净暴露决定结果，可能主要是方向配置；若费用吞噬增量，先减少无效交易可能比增加模型更直接。


In [ ]:
held = weights.shift(1).fillna(0)
asset_r = prices.pct_change().fillna(0)
starting_wealth = result.equity.shift(1).fillna(1)
amount_contributions = (held * asset_r).mul(starting_wealth, axis=0)
attribution = amount_contributions.sum()
attribution.loc["COST"] = -(result.cost * starting_wealth).sum()
print(attribution.sort_values().round(5))
print("贡献合计：", attribution.sum())
print("实际净值变化：", result.equity.iloc[-1] - 1)

yearly_rows = []
for year, group in result.groupby(result.index.year):
    row = summarize(group.net_return)
    row.update(year=year, observations=len(group), mean_turnover=group.turnover.mean())
    yearly_rows.append(row)
pd.DataFrame(yearly_rows).set_index("year").round(4)


逐年表是事后描述，不是把每年都重新调到最优的许可。只有某一年赚钱而其他年份都亏，或净值高度依赖单一资产，提示需要更具体的失败解释。可以预先定义趋势翻向、波动变高、资产同时下跌等状态，再报告全部分组和样本数；不能看到结果后只留下支持故事的状态。

考虑三个反例：价格快速反转时，过去 60 行上涨可能在转跌后才消失；共同因子冲击时，多个标签会同时亏损；真实市场流动性变差时，线性 10 bps 不足以描述退出。前两种可在生成数据中检查时序，第三种只能作为额外成本或成交约束情景，不能声称已有真实观测支持。

## 7. 手工练习与自查

### 练习 A：为什么单名上限会减少总暴露

原始分数为 `[4, 1, -1]`，先按绝对值和归一化，再逐项限制到 25%。写出最终权重、总暴露、净暴露；解释为什么不能再随意归一化。

<details><summary>参考答案</summary>

归一化为 `[2/3, 1/6, -1/6]`，截断为 `[1/4, 1/6, -1/6]`，总暴露为 7/12，净暴露 1/4。再归一化会提高第一项，可能重新突破上限；若要将余量分配到其他资产，需要明确另一个受约束配置算法，不能当成现有实现已经做了。

</details>

### 练习 B：同一天有收益和成本，它们用的是同一份目标吗

3 月 26 日首次目标总绝对权重约 0.954179，10 bps 费率。为什么该日毛收益为零、净收益为负？3 月 29 日为什么可以有正收益却零计划换手？

<details><summary>参考答案</summary>

3 月 26 日收益使用前一天零目标，新目标在当日形成并扣费，净收益为 `-0.954179 × 0.001`。3 月 29 日使用已经形成的目标赚取新一日收益，目标计划不变，所以 `weights.diff()` 为零。现实比例漂移仍可能需要交易，本实现没有计入它。

</details>

### 练习 C：怎样解释费用图

某窗口从 10 bps 提高到 25 bps，本日目标变更换手为 0.4。单期净收益下降多少？累计净值差是否等于逐日这些差相加？

<details><summary>参考答案</summary>

下降 `0.4 × 15 / 10000 = 0.0006`，即 6 个基点。净值由每日净收益连乘，费用变化还改变后续基数，因此累计差不能直接用日收益差相加；需要对两条路径分别复利。

</details>

### 练习 D：看过敏感性矩阵后，如何继续研究

你看到 20 行比 60 行结果更好，是否可以把“20 行有效”作为最终结论？下一个实验应该怎样设置？

<details><summary>参考答案</summary>

不能。矩阵已经参与选择，同一区间不能继续充当未触碰测试。先解释较短窗口是否通过更快响应改善结果，检查费用和暴露是否改变；再冻结规则，在尚未参与选择的时间或新数据上验证。换个合成 seed 可以检验对生成路径的稳定性，但不能证明真实市场规律。

</details>

### 自测：说清楚本实验最重要的三个边界

<details><summary>参考答案</summary>

数据是风险代理的合成路径，不是真实可交易合同；执行隐含理想收盘建仓，未模拟目标与实际持仓漂移、融资借券和订单成交；整段与矩阵展示是方法演示，未自动建立盲测。完整研究可以如实交付负结果，不能把这些边界用更多指标掩盖。

</details>

## 8. 导出一份与命令行一致的结果

最后用同一主配置调用入口，结果包含价格、目标、四种费用回放、基准、指标和清单。默认写入临时目录，便于重复运行而不在教材目录留下产物。要保存一项研究，先选择自己的独立输出目录，再记录配置、保留失败结果，并写成一页 memo：问题与机制、信息/成交时点、净结果与基准、风险、失败条件和下一步。

产物的 SHA-256 用于确认文件身份；它只能说明两人看到同一份文件，不能证明模型经济解释正确。交付时仍要能回到 3 月 26 日，把每一个分数、权重、费用和下一行收益讲清楚。


In [ ]:
LAB_OUTPUT = Path(tempfile.mkdtemp(prefix="alpha-research-lab-"))
exported_metrics = run_baseline(output_dir=LAB_OUTPUT)
print("实验导出目录：", LAB_OUTPUT)
print("导出文件：", sorted(p.name for p in LAB_OUTPUT.iterdir()))


## 9. 本实验小结

本实验从一日价格到一日目标，再到下一行损益，串起了完整主线。窗口决定看到多长的过去，波动缩放改变分数尺度，归一化和上限决定名义暴露，时间移位决定哪段收益属于这份计划，目标变化与费用决定毛净差，归因和分组说明结果在哪些条件下出现。

现在可以回到 README 的六个阶段开展自己的实验。先写预期，再只改变一项假设，最后保留能支持或推翻预期的完整证据。是否得到正收益不是完成标准；能否准确说明得到什么、为什么、尚未证明什么，才决定这个研究是否值得继续。
